In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir -p /content/drive/MyDrive/gpt2_spoc_model

In [3]:
!pip install pandas tqdm wget


  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=594b47bbbd07a03188dca309055d63f36ed5e66e288c4830bc6d305cce134f70
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [4]:
!mkdir -p data
!wget https://sumith1896.github.io/spoc/data/spoc.zip -O spoc.zip
!unzip -o spoc.zip -d data

!mv data/spoc data/spoc_dataset
!rm spoc.zip

--2025-11-02 06:32:44--  https://sumith1896.github.io/spoc/data/spoc.zip
Resolving sumith1896.github.io (sumith1896.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to sumith1896.github.io (sumith1896.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9960355 (9.5M) [application/x-zip-compressed]
Saving to: ‘spoc.zip’

spoc.zip            100%[===================>]   9.50M  --.-KB/s    in 0.07s   

2025-11-02 06:32:44 (133 MB/s) - ‘spoc.zip’ saved [9960355/9960355]

Archive:  spoc.zip
   creating: data/test/
  inflating: data/test/spoc-testw.tsv  
  inflating: data/test/spoc-testp.tsv  
   creating: data/testcases/
   creating: data/testcases/1000A/
  inflating: data/testcases/1000A/1000A_testcases.txt  
  inflating: data/testcases/1000A/1000A_testcases_hidden.txt  
  inflating: data/testcases/1000A/1000A_testcases_public.txt  
   creating: data/testcases/1003A/
  inflating: data/testcases/1003A/1003A_test

In [5]:
!ls data


LICENSE  README.md  test  testcases  train


In [6]:
!mkdir -p data/spoc_dataset
!mv data/train data/test data/testcases data/spoc_dataset/
!ls data/spoc_dataset/train/split


spoc-train-eval.tsv  spoc-train-test.tsv  spoc-train-train.tsv


In [7]:
import pandas as pd

train_path = "data/spoc_dataset/train/split/spoc-train-train.tsv"
eval_path = "data/spoc_dataset/train/split/spoc-train-eval.tsv"
test_path = "data/spoc_dataset/train/split/spoc-train-test.tsv"

df_train = pd.read_csv(train_path, sep='\t', header=None, names=['pseudo','code'])
df_eval = pd.read_csv(eval_path, sep='\t', header=None, names=['pseudo','code'])
df_test = pd.read_csv(test_path, sep='\t', header=None, names=['pseudo','code'])

print("✅ Train size:", len(df_train))
print("✅ Eval size:", len(df_eval))
print("✅ Test size:", len(df_test))
df_train.head()


/tmp/ipython-input-1798567749.py:7: DtypeWarning: Columns (2,4,5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(train_path, sep='\t', header=None, names=['pseudo','code'])


✅ Train size: 246087
✅ Eval size: 27289
✅ Test size: 20481


,,,,,pseudo,code
text,code,workerid,probid,subid,line,indent
NaN,int main() {,01,3A,41470897,0,0
create string s,string s;,01,3A,41470897,1,1
"create integers x1, y1, x2, y2","int x1, y1, x2, y2;",01,3A,41470897,2,1
read s,cin >> s;,01,3A,41470897,3,1


In [8]:
import pandas as pd

train_path = "data/spoc_dataset/train/split/spoc-train-train.tsv"

# Load with low_memory=False to suppress the warning
df_train_raw = pd.read_csv(train_path, sep='\t', low_memory=False)
print(df_train_raw.head(10))
print("\nColumns:", df_train_raw.columns.tolist())


                                                text  \
0                                                NaN   
1                                    create string s   
2                     create integers x1, y1, x2, y2   
3                                             read s   
4                                set x1 to s[0] - 96   
5                               set y1 to s[1] - '0'   
6                                             read s   
7                                set x2 to s[0] - 96   
8                               set y2 to s[1] - '0'   
9  print maximum of absolute value of x1 - x2 and...   

                                               code  workerid probid  \
0                                      int main() {         1     3A   
1                                         string s;         1     3A   
2                               int x1, y1, x2, y2;         1     3A   
3                                         cin >> s;         1     3A   
4                      

In [9]:
df_train = df_train_raw[['text', 'code']].dropna().reset_index(drop=True)
df_train.columns = ['pseudo', 'code']

print("✅ Cleaned Train size:", len(df_train))
df_train.head()


✅ Cleaned Train size: 181862


,pseudo,code
0,create string s,string s;
1,"create integers x1, y1, x2, y2","int x1, y1, x2, y2;"
2,read s,cin >> s;
3,set x1 to s[0] - 96,x1 = s[0] - 96;
4,set y1 to s[1] - '0',y1 = s[1] - '0';


In [10]:
df_train['formatted'] = (
    "### PSEUDOCODE:\n" + df_train['pseudo'].astype(str) +
    "\n### PYTHON CODE:\n" + df_train['code'].astype(str)
)

df_train['formatted'].to_csv("spoc_train_formatted.txt", index=False, header=False)

print("✅ Saved formatted dataset: spoc_train_formatted.txt")


✅ Saved formatted dataset: spoc_train_formatted.txt


In [11]:
# for converting to python
import pandas as pd
import re
import warnings
warnings.filterwarnings('ignore')


In [12]:
def cpp_to_python(cpp_code):
    """Convert C++ code to Python"""
    if pd.isna(cpp_code) or not cpp_code:
        return ""

    python_code = str(cpp_code).strip()

    # Variable declarations - Multiple variables
    def convert_multi_int(match):
        vars_str = match.group(1)
        vars_list = [v.strip() for v in vars_str.split(',')]
        return ', '.join(vars_list) + ' = ' + ', '.join(['0'] * len(vars_list))

    python_code = re.sub(r'\bint\s+([\w\s,]+);', convert_multi_int, python_code)

    # Single variable declarations
    python_code = re.sub(r'\bint\s+(\w+);', r'\1 = 0', python_code)
    python_code = re.sub(r'\bstring\s+(\w+);', r'\1 = ""', python_code)
    python_code = re.sub(r'\bchar\s+(\w+);', r'\1 = ""', python_code)
    python_code = re.sub(r'\b(double|float)\s+(\w+);', r'\2 = 0.0', python_code)
    python_code = re.sub(r'\blong\s+long\s+(\w+);', r'\1 = 0', python_code)
    python_code = re.sub(r'\bbool\s+(\w+);', r'\1 = False', python_code)
    python_code = re.sub(r'\bvector<[\w\s]+>\s+(\w+);', r'\1 = []', python_code)
    python_code = re.sub(r'\bint\s+(\w+)\[(\d+)\];', r'\1 = [0] * \2', python_code)

    # Input/Output - Multiple variables
    def convert_multi_cin(match):
        vars_str = match.group(1)
        vars_list = re.findall(r'\w+', vars_str)
        if len(vars_list) > 1:
            return ', '.join(vars_list) + ' = input().split()'
        else:
            return vars_list[0] + ' = input()'

    python_code = re.sub(r'\bcin\s*>>\s*([\w\s>>]+);', convert_multi_cin, python_code)

    # Output
    def convert_cout(match):
        content = match.group(1)
        content = content.replace('<< endl', '').replace('<<endl', '')
        parts = [p.strip() for p in content.split('<<') if p.strip()]
        return 'print(' + ', '.join(parts) + ')'

    python_code = re.sub(r'\bcout\s*<<\s*([^;]+);', convert_cout, python_code)

    # Control structures
    python_code = re.sub(
        r'\bfor\s*\(\s*int\s+(\w+)\s*=\s*(\d+)\s*;\s*\1\s*<\s*(\w+)\s*;\s*\1\+\+\s*\)',
        r'for \1 in range(\2, \3):', python_code
    )
    python_code = re.sub(
        r'\bfor\s*\(\s*int\s+(\w+)\s*=\s*(\d+)\s*;\s*\1\s*<=\s*(\w+)\s*;\s*\1\+\+\s*\)',
        r'for \1 in range(\2, \3 + 1):', python_code
    )
    python_code = re.sub(r'\bwhile\s*\(([^)]+)\)', r'while \1:', python_code)
    python_code = re.sub(r'\bif\s*\(([^)]+)\)', r'if \1:', python_code)
    python_code = re.sub(r'\belse\s+if\s*\(([^)]+)\)', r'elif \1:', python_code)
    python_code = re.sub(r'\belse\s*{', r'else:', python_code)

    # Operators
    python_code = python_code.replace('&&', ' and ')
    python_code = python_code.replace('||', ' or ')
    python_code = re.sub(r'!(\w+)', r'not \1', python_code)
    python_code = python_code.replace(' true', ' True')
    python_code = python_code.replace(' false', ' False')

    # String operations
    python_code = re.sub(r'(\w+)\.length\(\)', r'len(\1)', python_code)
    python_code = re.sub(r'(\w+)\.size\(\)', r'len(\1)', python_code)
    python_code = re.sub(r'\.push_back\(', r'.append(', python_code)

    # Cleanup
    python_code = python_code.replace(';', '')
    python_code = python_code.replace('{', '').replace('}', '')
    python_code = re.sub(r'\s+', ' ', python_code)

    return python_code.strip()

print("✅ Converter function loaded!")

✅ Converter function loaded!


In [13]:
# load and test convversion
# Load the dataset
print("📂 Loading dataset...")
df = pd.read_csv("data/spoc_dataset/train/split/spoc-train-train.tsv",
                 sep='\t', low_memory=False)

# Clean and prepare
df = df[['text', 'code']].dropna().reset_index(drop=True)
df.columns = ['pseudo', 'cpp_code']

print(f"✅ Loaded {len(df)} samples")

# Test conversion on first 10 samples
print("\n🧪 Testing conversion on 10 samples...")
print("=" * 60)

for i in range(10):
    pseudo = df['pseudo'].iloc[i]
    cpp = df['cpp_code'].iloc[i]
    python = cpp_to_python(cpp)

    print(f"\n--- Example {i+1} ---")
    print(f"Pseudo: {pseudo}")
    print(f"C++:    {cpp}")
    print(f"Python: {python}")
    print("-" * 40)

print("\n✅ If conversions look good, proceed to next cell!")

📂 Loading dataset...
✅ Loaded 181862 samples

🧪 Testing conversion on 10 samples...

--- Example 1 ---
Pseudo: create string s
C++:    string s;
Python: s = ""
----------------------------------------

--- Example 2 ---
Pseudo: create integers x1, y1, x2, y2
C++:    int x1, y1, x2, y2;
Python: x1, y1, x2, y2 = 0, 0, 0, 0
----------------------------------------

--- Example 3 ---
Pseudo: read s
C++:    cin >> s;
Python: s = input()
----------------------------------------

--- Example 4 ---
Pseudo: set x1 to s[0] - 96
C++:    x1 = s[0] - 96;
Python: x1 = s[0] - 96
----------------------------------------

--- Example 5 ---
Pseudo: set y1 to s[1] - '0'
C++:    y1 = s[1] - '0';
Python: y1 = s[1] - '0'
----------------------------------------

--- Example 6 ---
Pseudo: read s
C++:    cin >> s;
Python: s = input()
----------------------------------------

--- Example 7 ---
Pseudo: set x2 to s[0] - 96
C++:    x2 = s[0] - 96;
Python: x2 = s[0] - 96
----------------------------------------

-

In [14]:
#Convert Full Dataset (8000 samples)
print("🔄 Converting 8000 samples to Python...")

# Take 8000 samples for training
df_train = df.head(8000).copy()

# Convert C++ to Python
df_train['python_code'] = df_train['cpp_code'].apply(cpp_to_python)

# Create formatted training data
df_train['formatted'] = (
    "### PSEUDOCODE:\n" + df_train['pseudo'].astype(str) +
    "\n### PYTHON CODE:\n" + df_train['python_code'].astype(str)
)

print(f"✅ Converted {len(df_train)} samples!")

# Show statistics
print("\n📊 Conversion Statistics:")
changed = sum(df_train['cpp_code'] != df_train['python_code'])
print(f"   Modified: {changed}/{len(df_train)} ({changed/len(df_train)*100:.1f}%)")
print(f"   Avg C++ length: {df_train['cpp_code'].str.len().mean():.1f} chars")
print(f"   Avg Python length: {df_train['python_code'].str.len().mean():.1f} chars")

# Preview first 3 samples
print("\n📝 First 3 converted samples:")
print("=" * 60)
for i in range(3):
    print(f"\n{df_train['formatted'].iloc[i]}")
    print("-" * 40)

🔄 Converting 8000 samples to Python...
✅ Converted 8000 samples!

📊 Conversion Statistics:
   Modified: 7873/8000 (98.4%)
   Avg C++ length: 19.0 chars
   Avg Python length: 16.9 chars

📝 First 3 converted samples:

### PSEUDOCODE:
create string s
### PYTHON CODE:
s = ""
----------------------------------------

### PSEUDOCODE:
create integers x1, y1, x2, y2
### PYTHON CODE:
x1, y1, x2, y2 = 0, 0, 0, 0
----------------------------------------

### PSEUDOCODE:
read s
### PYTHON CODE:
s = input()
----------------------------------------


In [15]:
# Save the converted dataset
OUTPUT_PATH = "spoc_train_python.txt"

print(f"💾 Saving to {OUTPUT_PATH}...")

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for text in df_train['formatted']:
        f.write(text + '\n\n')  # Double newline between samples

print(f"✅ Saved {len(df_train)} samples to {OUTPUT_PATH}")

# Verify file size
import os
file_size = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)  # MB
print(f"   File size: {file_size:.2f} MB")

# Show a sample of what was saved
print("\n📄 First few lines of saved file:")
with open(OUTPUT_PATH, 'r') as f:
    print(f.read(500))

💾 Saving to spoc_train_python.txt...
✅ Saved 8000 samples to spoc_train_python.txt
   File size: 0.59 MB

📄 First few lines of saved file:
### PSEUDOCODE:
create string s
### PYTHON CODE:
s = ""

### PSEUDOCODE:
create integers x1, y1, x2, y2
### PYTHON CODE:
x1, y1, x2, y2 = 0, 0, 0, 0

### PSEUDOCODE:
read s
### PYTHON CODE:
s = input()

### PSEUDOCODE:
set x1 to s[0] - 96
### PYTHON CODE:
x1 = s[0] - 96

### PSEUDOCODE:
set y1 to s[1] - '0'
### PYTHON CODE:
y1 = s[1] - '0'

### PSEUDOCODE:
read s
### PYTHON CODE:
s = input()

### PSEUDOCODE:
set x2 to s[0] - 96
### PYTHON CODE:
x2 = s[0] - 96

### PSEUDOCODE:
set y2 to s[1] - '0


# Fine-Tune GPT-2

In [16]:
"""
CORRECTED GPT-2 FINE-TUNING WITH C++ TO PYTHON CONVERSION
==========================================================
This script properly converts C++ to Python BEFORE training
"""

# ========================
# SECTION 1: SETUP & INSTALLATION
# ========================

print("📦 Installing required packages...")
!pip install transformers datasets accelerate torch evaluate -q

import os
import pandas as pd
import torch
import re
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset, DatasetDict
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✅ Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# ========================
# SECTION 2: C++ TO PYTHON CONVERTER
# ========================

def cpp_to_python(cpp_code):
    """
    Convert C++ code to Python equivalent
    Handles common C++ patterns in SPOC dataset
    """
    if pd.isna(cpp_code) or not cpp_code:
        return ""

    python_code = str(cpp_code).strip()

    # === VARIABLE DECLARATIONS ===

    # Multiple int declarations: int x, y, z;
    def convert_multi_int(match):
        vars_str = match.group(1)
        vars_list = [v.strip() for v in vars_str.split(',')]
        return ', '.join(vars_list) + ' = ' + ', '.join(['0'] * len(vars_list))

    python_code = re.sub(r'\bint\s+([\w\s,]+);', convert_multi_int, python_code)

    # Single variable declarations
    python_code = re.sub(r'\bint\s+(\w+);', r'\1 = 0', python_code)
    python_code = re.sub(r'\bstring\s+(\w+);', r'\1 = ""', python_code)
    python_code = re.sub(r'\bchar\s+(\w+);', r'\1 = ""', python_code)
    python_code = re.sub(r'\b(double|float)\s+(\w+);', r'\2 = 0.0', python_code)
    python_code = re.sub(r'\blong\s+long\s+(\w+);', r'\1 = 0', python_code)
    python_code = re.sub(r'\bbool\s+(\w+);', r'\1 = False', python_code)

    # Arrays and vectors
    python_code = re.sub(r'\bvector<[\w\s]+>\s+(\w+);', r'\1 = []', python_code)
    python_code = re.sub(r'\bint\s+(\w+)\[(\d+)\];', r'\1 = [0] * \2', python_code)

    # === INPUT/OUTPUT ===

    # Multiple cin: cin >> x >> y >> z;
    def convert_multi_cin(match):
        vars_str = match.group(1)
        vars_list = re.findall(r'\w+', vars_str)
        if len(vars_list) > 1:
            return ', '.join(vars_list) + ' = input().split()'
        else:
            return vars_list[0] + ' = input()'

    python_code = re.sub(r'\bcin\s*>>\s*([\w\s>>]+);', convert_multi_cin, python_code)

    # cout with multiple outputs
    def convert_cout(match):
        content = match.group(1)
        content = content.replace('<< endl', '').replace('<<endl', '')
        parts = [p.strip() for p in content.split('<<') if p.strip()]
        return 'print(' + ', '.join(parts) + ')'

    python_code = re.sub(r'\bcout\s*<<\s*([^;]+);', convert_cout, python_code)

    # === CONTROL STRUCTURES ===

    # for loops
    python_code = re.sub(
        r'\bfor\s*\(\s*int\s+(\w+)\s*=\s*(\d+)\s*;\s*\1\s*<\s*(\w+)\s*;\s*\1\+\+\s*\)',
        r'for \1 in range(\2, \3):',
        python_code
    )
    python_code = re.sub(
        r'\bfor\s*\(\s*int\s+(\w+)\s*=\s*(\d+)\s*;\s*\1\s*<=\s*(\w+)\s*;\s*\1\+\+\s*\)',
        r'for \1 in range(\2, \3 + 1):',
        python_code
    )

    # while, if, else
    python_code = re.sub(r'\bwhile\s*\(([^)]+)\)', r'while \1:', python_code)
    python_code = re.sub(r'\bif\s*\(([^)]+)\)', r'if \1:', python_code)
    python_code = re.sub(r'\belse\s+if\s*\(([^)]+)\)', r'elif \1:', python_code)
    python_code = re.sub(r'\belse\s*{?', r'else:', python_code)

    # === OPERATORS ===

    python_code = python_code.replace('&&', ' and ')
    python_code = python_code.replace('||', ' or ')
    python_code = re.sub(r'!(\w+)', r'not \1', python_code)
    python_code = python_code.replace(' true', ' True')
    python_code = python_code.replace(' false', ' False')

    # === STRING/ARRAY OPERATIONS ===

    python_code = re.sub(r'(\w+)\.length\(\)', r'len(\1)', python_code)
    python_code = re.sub(r'(\w+)\.size\(\)', r'len(\1)', python_code)
    python_code = re.sub(r'\.push_back\(', r'.append(', python_code)

    # === CLEANUP ===

    python_code = python_code.replace(';', '')
    python_code = python_code.replace('{', '').replace('}', '')
    python_code = re.sub(r'\s+', ' ', python_code)

    return python_code.strip()

print("✅ C++ to Python converter loaded!")

# ========================
# SECTION 3: CONFIGURATION
# ========================

class Config:
    TRAIN_PATH = "data/spoc_dataset/train/split/spoc-train-train.tsv"
    OUTPUT_DIR = "/content/drive/MyDrive/gpt2_python_correct_small"

    TRAIN_SIZE = 8000
    EVAL_SIZE = 2000
    MAX_LENGTH = 256
    MODEL_NAME = "gpt2"

    BATCH_SIZE = 4          # doubled
    GRADIENT_ACCUMULATION = 4
    LEARNING_RATE = 5e-5
    NUM_EPOCHS = 2          # increase to 2
    WARMUP_STEPS = 200

    EVAL_STEPS = 500
    SAVE_STEPS = 500
    LOGGING_STEPS = 100


config = Config()

# ========================
# SECTION 4: LOAD & CONVERT DATASET
# ========================

def load_and_convert_dataset(train_path, config):
    """Load dataset and convert C++ to Python"""
    print("\n📊 Loading dataset...")

    # Load data
    df = pd.read_csv(train_path, sep='\t', low_memory=False)
    df = df[['text', 'code']].dropna().reset_index(drop=True)
    df.columns = ['pseudo', 'cpp_code']

    print(f"   Total samples: {len(df)}")

    # Convert C++ to Python
    print("\n🔄 Converting C++ to Python...")
    df['python_code'] = df['cpp_code'].apply(cpp_to_python)

    # Show conversion examples
    print("\n📝 Sample Conversions:")
    print("=" * 60)
    for i in range(min(5, len(df))):
        print(f"\nExample {i+1}:")
        print(f"  Pseudo: {df['pseudo'].iloc[i]}")
        print(f"  C++:    {df['cpp_code'].iloc[i]}")
        print(f"  Python: {df['python_code'].iloc[i]}")
        print("-" * 40)

    # Count conversions
    changed = sum(df['cpp_code'] != df['python_code'])
    print(f"\n✅ Converted: {changed}/{len(df)} samples ({changed/len(df)*100:.1f}%)")

    # Format data with PYTHON CODE label
    df['formatted'] = (
        "### PSEUDOCODE:\n" + df['pseudo'].astype(str) +
        "\n### PYTHON CODE:\n" + df['python_code'].astype(str)  # ← Now actually Python!
    )

    # Create train/eval split
    total_size = config.TRAIN_SIZE + config.EVAL_SIZE
    df_subset = df.head(total_size)

    train_data = df_subset['formatted'][:config.TRAIN_SIZE].tolist()
    eval_data = df_subset['formatted'][config.TRAIN_SIZE:].tolist()

    # Create dataset
    dataset = DatasetDict({
        'train': Dataset.from_dict({'text': train_data}),
        'eval': Dataset.from_dict({'text': eval_data})
    })

    print(f"\n✅ Train size: {len(dataset['train'])}")
    print(f"✅ Eval size: {len(dataset['eval'])}")

    return dataset

# Load and convert dataset
dataset = load_and_convert_dataset(config.TRAIN_PATH, config)

# Show sample
print("\n📝 Sample training data:")
print(dataset['train'][0]['text'][:300] + "...")

# ========================
# SECTION 5: MODEL & TOKENIZER
# ========================

print("\n🤖 Loading GPT-2 model and tokenizer...")

tokenizer = GPT2Tokenizer.from_pretrained(config.MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(config.MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

print(f"✅ Model loaded: {model.num_parameters():,} parameters")

# ========================
# SECTION 6: TOKENIZATION
# ========================

print("\n🔤 Tokenizing dataset...")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=config.MAX_LENGTH,
        return_tensors=None
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing"
)

print("✅ Tokenization complete")

# ========================
# SECTION 7: TRAINING SETUP
# ========================

print("\n⚙️ Setting up training...")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir=config.OUTPUT_DIR,
    num_train_epochs=config.NUM_EPOCHS,
    per_device_train_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRADIENT_ACCUMULATION,
    learning_rate=config.LEARNING_RATE,
    warmup_steps=config.WARMUP_STEPS,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=config.EVAL_STEPS,
    per_device_eval_batch_size=config.BATCH_SIZE,
    save_strategy="steps",
    save_steps=config.SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_steps=config.LOGGING_STEPS,
    logging_dir=f"{config.OUTPUT_DIR}/logs",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none",
    seed=42
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("✅ Training setup complete")

# ========================
# SECTION 8: TRAINING
# ========================

print("\n🚀 Starting training...")
print("=" * 60)

train_result = trainer.train()

print("\n" + "=" * 60)
print("📊 TRAINING RESULTS")
print("=" * 60)
print(f"Final Loss: {train_result.metrics.get('train_loss', 'N/A'):.4f}")
print(f"Training Time: {train_result.metrics.get('train_runtime', 0):.2f}s")
print("=" * 60)

# ========================
# SECTION 9: SAVE MODEL
# ========================

print("\n💾 Saving model...")

final_path = f"{config.OUTPUT_DIR}/final"
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

print(f"✅ Model saved to: {final_path}")

# ========================
# SECTION 10: TEST PYTHON GENERATION
# ========================

print("\n🧪 Testing PYTHON code generation...")
print("=" * 60)

model.eval()

test_cases = [
    "create integer variable x",
    "read input from user",
    "for i from 0 to 10 print i",
    "if x greater than 5 print yes",
    "create list numbers"
]

def generate_code(pseudo_code):
    prompt = f"### PSEUDOCODE:\n{pseudo_code}\n### PYTHON CODE:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### PYTHON CODE:" in generated:
        code = generated.split("### PYTHON CODE:")[1].strip()
        if "###" in code:
            code = code.split("###")[0].strip()
        return code

    return generated

print("\n🐍 Generated PYTHON Code:")
print("=" * 60)

for i, test in enumerate(test_cases, 1):
    print(f"\n--- Test {i} ---")
    print(f"Pseudo: {test}")
    code = generate_code(test)
    print(f"Python: {code}")

    # Check if it's actually Python-like
    is_python = not any(cpp_keyword in code for cpp_keyword in ['int ', 'string ', 'cin', 'cout', '<<'])
    print(f"Looks like Python: {'✅' if is_python else '❌ (still has C++ syntax)'}")
    print("-" * 40)

# ========================
# SECTION 11: EVALUATION METRICS
# ========================

print("\n📈 Computing evaluation metrics...")

eval_results = trainer.evaluate()

print("\n" + "=" * 60)
print("📊 EVALUATION METRICS")
print("=" * 60)
print(f"Eval Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity: {torch.exp(torch.tensor(eval_results['eval_loss'])):.2f}")
print("=" * 60)

# ========================
# SECTION 12: SUMMARY
# ========================

print("\n" + "=" * 60)
print("🎉 TRAINING COMPLETE!")
print("=" * 60)
print("\n✅ What we did differently:")
print("   1. Converted C++ code to Python BEFORE training")
print("   2. Model learned actual Python syntax")
print("   3. Now generates Python, not C++!")
print("\n📁 Model saved to:")
print(f"   {final_path}")
print("\n📥 To download:")
print(f"   !zip -r gpt2-python-model.zip {final_path}")
print(f"   from google.colab import files")
print(f"   files.download('gpt2-python-model.zip')")
print("=" * 60)

print("\n✅ Your model now generates ACTUAL PYTHON code! 🐍🎉")

📦 Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00

✅ Using device: cuda
   GPU: Tesla T4
   Memory: 15.83 GB
✅ C++ to Python converter loaded!

📊 Loading dataset...
   Total samples: 181862

🔄 Converting C++ to Python...

📝 Sample Conversions:

Example 1:
  Pseudo: create string s
  C++:    string s;
  Python: s = ""
----------------------------------------

Example 2:
  Pseudo: create integers x1, y1, x2, y2
  C++:    int x1, y1, x2, y2;
  Python: x1, y1, x2, y2 = 0, 0, 0, 0
----------------------------------------

Example 3:
  Pseudo: read s
  C++:    cin >> s;
  Python: s = input()
----------------------------------------

Example 4:
  Pseudo: set x1 to s[0] - 96
  C++:    x1 = s[0] - 96;
  Python: x1 = s[0] - 96
----------------------------------------

Example 5:
  Pseudo: set y1 to s[1] - '0'
  C++:    y1 = s[1] - '0';
  Python: y1 = s[1] - '0'
----------------------------------------

✅ Converted: 181421/181862 sampl

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,0.595000,0.825015
1000,0.487900,0.790036


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



📊 TRAINING RESULTS
Final Loss: 0.7533
Training Time: 563.51s

💾 Saving model...
✅ Model saved to: /content/drive/MyDrive/gpt2_python_correct_small/final

🧪 Testing PYTHON code generation...

🐍 Generated PYTHON Code:

--- Test 1 ---
Pseudo: create integer variable x
Python: x = 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, PYTHON CODE:
Looks like Python: ✅
----------------------------------------

--- Test 2 ---
Pseudo: read input from user
Python: input = input() #'\n'
Looks like Python: ✅
----------------------------------------

--- Test 3 ---
Pseudo: for i from 0 to 10 print i
Python: for (int i = 0 i < 10 i++) print(i)
Looks like Python: ❌ (still has C++ syntax)
----------------------------------------

--- Test 4 ---
Pseudo: if x greater than 5 print yes
Python: if x > 5: print("YES") else: print("NO") break
Looks like Python: ✅
-------------------


📊 EVALUATION METRICS
Eval Loss: 0.7900
Perplexity: 2.20

🎉 TRAINING COMPLETE!

✅ What we did differently:
   1. Converted C++ code to Python BEFORE training
   2. Model learned actual Python syntax
   3. Now generates Python, not C++!

📁 Model saved to:
   /content/drive/MyDrive/gpt2_python_correct_small/final

📥 To download:
   !zip -r gpt2-python-model.zip /content/drive/MyDrive/gpt2_python_correct_small/final
   from google.colab import files
   files.download('gpt2-python-model.zip')

✅ Your model now generates ACTUAL PYTHON code! 🐍🎉


# Evalutaion

In [19]:
!ls /content/drive/MyDrive/gpt2_python_correct_small/final


config.json		model.safetensors	 training_args.bin
generation_config.json	special_tokens_map.json  vocab.json
merges.txt		tokenizer_config.json


In [21]:
"""
COLAB-OPTIMIZED EVALUATION SCRIPT
===================================
Lightweight evaluation for limited resources
"""

# ========================
# SETUP
# ========================

!pip install sacrebleu evaluate -q

import pandas as pd
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from tqdm import tqdm
import evaluate
import json

# ========================
# LOAD MODEL
# ========================

print("🔄 Loading trained model...")

MODEL_PATH = "/content/drive/MyDrive/gpt2_python_correct_small/final"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_PATH)
model = GPT2LMHeadModel.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

print(f"✅ Model loaded on {device}")

# ========================
# LOAD TEST DATA
# ========================

print("\n📊 Loading test data...")

TEST_PATH = "data/spoc_dataset/train/split/spoc-train-test.tsv"
NUM_SAMPLES = 200  # Reduced for Colab

df_test = pd.read_csv(TEST_PATH, sep='\t', low_memory=False)
df_test = df_test[['text', 'code']].dropna().reset_index(drop=True)
df_test.columns = ['pseudo', 'code']
df_test = df_test.head(NUM_SAMPLES)

print(f"✅ Loaded {len(df_test)} test samples")

# ========================
# GENERATE PREDICTIONS
# ========================

print("\n🤖 Generating predictions...")

predictions = []
references = df_test['code'].tolist()

for pseudo in tqdm(df_test['pseudo'], desc="Generating"):
    prompt = f"### PSEUDOCODE:\n{pseudo}\n### PYTHON CODE:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### PYTHON CODE:" in generated:
        code = generated.split("### PYTHON CODE:")[1].strip()
        if "###" in code:
            code = code.split("###")[0].strip()
        predictions.append(code)
    else:
        predictions.append(generated)

print(f"✅ Generated {len(predictions)} predictions")

# ========================
# COMPUTE METRICS
# ========================

print("\n📈 Computing metrics...")

# 1. BLEU Score
bleu = evaluate.load("sacrebleu")
formatted_refs = [[ref] for ref in references]
bleu_results = bleu.compute(predictions=predictions, references=formatted_refs)

# 2. Exact Match
exact_matches = sum(p.strip() == r.strip() for p, r in zip(predictions, references))
exact_match_acc = exact_matches / len(predictions) * 100

# 3. Syntax Accuracy
valid_syntax = 0
for code in predictions:
    try:
        compile(code, '<string>', 'exec')
        valid_syntax += 1
    except:
        pass
syntax_acc = valid_syntax / len(predictions) * 100

# 4. Token Accuracy
correct_tokens = 0
total_tokens = 0

for pred, ref in zip(predictions, references):
    pred_tokens = tokenizer.encode(pred)
    ref_tokens = tokenizer.encode(ref)

    min_len = min(len(pred_tokens), len(ref_tokens))
    correct_tokens += sum(p == r for p, r in zip(pred_tokens[:min_len], ref_tokens[:min_len]))
    total_tokens += max(len(pred_tokens), len(ref_tokens))

token_acc = correct_tokens / total_tokens * 100 if total_tokens > 0 else 0

# ========================
# DISPLAY RESULTS
# ========================

print("\n" + "=" * 60)
print("📊 EVALUATION RESULTS")
print("=" * 60)
print(f"Samples Evaluated: {len(predictions)}")
print(f"\nBLEU Score: {bleu_results['score']:.2f}")
print(f"Exact Match: {exact_match_acc:.2f}%")
print(f"Syntax Accuracy: {syntax_acc:.2f}%")
print(f"Token Accuracy: {token_acc:.2f}%")
print("=" * 60)

# ========================
# SAVE RESULTS
# ========================

print("\n💾 Saving results...")

# Save metrics
metrics = {
    "bleu": bleu_results['score'],
    "exact_match": exact_match_acc,
    "syntax_accuracy": syntax_acc,
    "token_accuracy": token_acc,
    "num_samples": len(predictions)
}

with open("colab_evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save predictions
results_df = pd.DataFrame({
    'pseudo_code': df_test['pseudo'],
    'reference': references,
    'prediction': predictions
})
results_df.to_csv("colab_evaluation_results.csv", index=False)

print("✅ Results saved:")
print("   - colab_evaluation_metrics.json")
print("   - colab_evaluation_results.csv")

# ========================
# SHOW SAMPLE PREDICTIONS
# ========================

print("\n" + "=" * 60)
print("📝 SAMPLE PREDICTIONS")
print("=" * 60)

# Show 5 random samples
samples = results_df.sample(n=min(5, len(results_df)))

for idx, row in samples.iterrows():
    is_match = row['reference'].strip() == row['prediction'].strip()

    print(f"\n--- Sample {idx} ---")
    print(f"Pseudo: {row['pseudo_code']}")
    print(f"Ref:    {row['reference']}")
    print(f"Pred:   {row['prediction']}")
    print(f"Match:  {'✅' if is_match else '❌'}")
    print("-" * 40)

# ========================
# ANALYSIS
# ========================

print("\n" + "=" * 60)
print("🔍 ANALYSIS")
print("=" * 60)

# Calculate average lengths
avg_ref_len = sum(len(r) for r in references) / len(references)
avg_pred_len = sum(len(p) for p in predictions) / len(predictions)

print(f"Average Reference Length: {avg_ref_len:.1f} chars")
print(f"Average Prediction Length: {avg_pred_len:.1f} chars")
print(f"Length Ratio: {avg_pred_len/avg_ref_len:.2f}")

# Count perfect matches
perfect_matches = sum(1 for p, r in zip(predictions, references) if p.strip() == r.strip())
print(f"\nPerfect Matches: {perfect_matches} ({perfect_matches/len(predictions)*100:.1f}%)")

# Count syntax errors
syntax_errors = len(predictions) - valid_syntax
print(f"Syntax Errors: {syntax_errors} ({syntax_errors/len(predictions)*100:.1f}%)")

print("=" * 60)

# ========================
# DOWNLOAD RESULTS
# ========================

print("\n📥 To download results, run:")
print("""
from google.colab import files
files.download('colab_evaluation_metrics.json')
files.download('colab_evaluation_results.csv')
""")

print("\n✅ Evaluation complete! 🎉")

🔄 Loading trained model...
✅ Model loaded on cuda

📊 Loading test data...
✅ Loaded 200 test samples

🤖 Generating predictions...


Generating: 100%|██████████| 200/200 [04:06<00:00,  1.23s/it]


✅ Generated 200 predictions

📈 Computing metrics...



📊 EVALUATION RESULTS
Samples Evaluated: 200

BLEU Score: 10.55
Exact Match: 0.00%
Syntax Accuracy: 38.50%
Token Accuracy: 5.40%

💾 Saving results...
✅ Results saved:
   - colab_evaluation_metrics.json
   - colab_evaluation_results.csv

📝 SAMPLE PREDICTIONS

--- Sample 95 ---
Pseudo: read n
Ref:    cin >> n;
Pred:   n = input().split() + 1
Match:  ❌
----------------------------------------

--- Sample 15 ---
Pseudo: print integer casted size of ans print newline
Ref:    cout << (int)ans.size() << endl;
Pred:   print(int(ans))
Match:  ❌
----------------------------------------

--- Sample 30 ---
Pseudo: print count + 1
Ref:    cout << count + 1 << endl;
Pred:   print(count + 1)
Match:  ❌
----------------------------------------

--- Sample 158 ---
Pseudo: else
Ref:    else
Pred:   else:if s[1] is less than t[1]: break
Match:  ❌
----------------------------------------

--- Sample 128 ---
Pseudo: create integer n
Ref:    int n;
Pred:   n = 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 